# Lab 51 — Códigos qLDPC y Decodificador Neuronal

Los códigos **qLDPC** (*quantum Low-Density Parity-Check*) son el candidato principal a reemplazar los surface codes en procesadores cuánticos tolerantes a fallos a gran escala. IBM Nature 2024 demostró que los códigos bivariate bicycle permiten reducir el overhead de qubits físicos en un factor ~10× respecto al surface code para la misma capacidad lógica.

En este laboratorio construiremos:
1. Un código CSS con su matriz de paridad y verificación de la condición CSS
2. Cálculo y visualización de síndromes
3. Decodificador MWPM simplificado con NetworkX
4. Decodificador neuronal (MLP) entrenado con errores sintéticos
5. Curva de umbral: tasa de error lógico vs tasa de error físico

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from itertools import product
np.random.seed(42)

print('NumPy:', np.__version__)
print('NetworkX:', nx.__version__)

## 1. Códigos CSS — Construcción y Verificación

Un **código CSS** [[n, k, d]] se define por dos matrices binarias $H_X, H_Z \in \mathbb{F}_2^{r\times n}$ que satisfacen:
$$H_X H_Z^T = 0 \pmod{2}$$

El código de Steane [[7,1,3]] usa la matriz de paridad del código de Hamming [7,4,3]:

$$H = \begin{pmatrix} 1&0&1&0&1&0&1 \\ 0&1&1&0&0&1&1 \\ 0&0&0&1&1&1&1 \end{pmatrix}$$

In [ ]:
# Código de Steane [[7,1,3]]: H_X = H_Z = matriz Hamming
H_steane = np.array([
    [1,0,1,0,1,0,1],
    [0,1,1,0,0,1,1],
    [0,0,0,1,1,1,1]
], dtype=int)

H_X = H_steane.copy()
H_Z = H_steane.copy()
n_qubits = H_X.shape[1]
k_logical = n_qubits - np.linalg.matrix_rank(H_X) - np.linalg.matrix_rank(H_Z)

# Verificar condición CSS
CSS_check = (H_X @ H_Z.T) % 2
print(f"[[{n_qubits}, {k_logical}, 3]] — Código de Steane")
print(f"H_X @ H_Z^T (mod 2) = {CSS_check.tolist()} ({'✓ CSS válido' if np.all(CSS_check==0) else '✗ Error'})")

# Código repetición [[3,1,1]] como CSS simple para ilustración
H_rep = np.array([[1,1,0],[0,1,1]], dtype=int)
H_rep_Z = np.array([[1,1,0],[0,1,1]], dtype=int)
print(f"\nCódigo repetición [[3,1,1]]: CSS check = {((H_rep @ H_rep_Z.T) % 2).tolist()}")

## 2. Síndromes de Error

Un error de Pauli $E_X$ (flip de bit) produce síndrome $s = H_Z E_X^T \pmod{2}$.
El síndrome no revela la posición del error lógico, solo el patrón de violación de estabilizadores.

Para el código de Steane, un error en el qubit $i$ produce el síndrome $= i$-ésima columna de $H_Z$.

In [ ]:
def compute_syndrome(H, error):
    """Calcula síndrome s = H e^T (mod 2)"""
    return (H @ error) % 2

def apply_random_error(n, p):
    """Genera vector de error con probabilidad p por qubit."""
    return (np.random.random(n) < p).astype(int)

# Demostración: error en qubit 3
for error_pos in [0, 2, 5]:
    e = np.zeros(n_qubits, dtype=int)
    e[error_pos] = 1
    syn = compute_syndrome(H_Z, e)
    print(f"Error en qubit {error_pos}: síndrome = {syn}")

# Tabla de lookup síndrome → qubit
print("\nTabla de síndromes Steane [[7,1,3]]:")
syndrome_table = {}
for i in range(n_qubits):
    e = np.zeros(n_qubits, dtype=int)
    e[i] = 1
    s = tuple(compute_syndrome(H_Z, e))
    syndrome_table[s] = i
    print(f"  síndrome {list(s)} → qubit {i}")

## 3. MWPM — Minimum Weight Perfect Matching

Para el **surface code** y códigos CSS generales, el decodificador MWPM construye un grafo donde:
- **Nodos** = estabilizadores violados (síndrome activo)
- **Aristas** = pares con peso = distancia mínima entre ellos
- **Matching** = emparejamiento de peso mínimo → corrección propuesta

In [ ]:
def syndrome_to_positions(syndrome):
    """Retorna posiciones de estabilizadores violados (síndrome=1)."""
    return [i for i, s in enumerate(syndrome) if s == 1]

def mwpm_decode(H, syndrome, qubit_positions=None):
    """
    MWPM simplificado para código 1D (cadena de estabilizadores).
    Construye grafo completo entre defectos y busca matching óptimo.
    """
    defects = syndrome_to_positions(syndrome)
    if len(defects) == 0:
        return np.zeros(H.shape[1], dtype=int)
    if len(defects) % 2 != 0:
        defects.append(H.shape[0])  # ancla al borde
    
    G = nx.Graph()
    for i, d1 in enumerate(defects):
        for j, d2 in enumerate(defects):
            if i < j:
                G.add_edge(i, j, weight=abs(d1 - d2))
    
    matching = nx.min_weight_matching(G)
    correction = np.zeros(H.shape[1], dtype=int)
    for i, j in matching:
        d1, d2 = sorted([defects[i], defects[j]])
        for q in range(min(d1+1, H.shape[1]), min(d2+1, H.shape[1])):
            correction[q] ^= 1
    return correction

# Prueba con código de repetición para cadena 1D de 7 qubits
H_chain = np.zeros((6, 7), dtype=int)
for i in range(6):
    H_chain[i, i] = 1
    H_chain[i, i+1] = 1

# Error en qubit 3
e_test = np.array([0,0,0,1,0,0,0])
syn_test = compute_syndrome(H_chain, e_test)
correction = mwpm_decode(H_chain, syn_test)
residual = (e_test + correction) % 2
print(f"Error:      {e_test}")
print(f"Síndrome:   {syn_test}")
print(f"Corrección: {correction}")
print(f"Residual:   {residual} ({'✓ Corregido' if np.all(residual==0) else '✗ Error lógico'})")

## 4. Decodificador Neuronal (MLP)

Un **decodificador neuronal** recibe el síndrome como entrada y predice la clase de error (sin error / error lógico). Para entrenarlo generamos un dataset sintético de síndromes con sus etiquetas de error lógico.

In [ ]:
def generate_training_data(H_X, H_Z, n_samples=10000, p_error=0.05):
    """
    Genera pares (síndrome, error_lógico) para entrenamiento.
    Error lógico = 1 si el operador lógico X_L tiene solapamiento impar con el error.
    Para código repetición: X_L = todos los qubits → error lógico si peso(e) es impar.
    """
    n = H_X.shape[1]
    r = H_X.shape[0]
    X_data = np.zeros((n_samples, r), dtype=int)
    y_data = np.zeros(n_samples, dtype=int)
    
    for i in range(n_samples):
        error = apply_random_error(n, p_error)
        syndrome = compute_syndrome(H_X, error)
        # Error lógico: proyección sobre operador lógico (peso total en código 1D)
        logical_error = int(np.sum(error) % 2)
        X_data[i] = syndrome
        y_data[i] = logical_error
    
    return X_data, y_data

# Usar código cadena 7 qubits
X_train_raw, y_train_raw = generate_training_data(H_chain, H_chain, n_samples=15000, p_error=0.05)
X_tr, X_te, y_tr, y_te = train_test_split(X_train_raw, y_train_raw, test_size=0.2, random_state=42)

# Entrenar MLP
mlp = MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu',
                    max_iter=200, random_state=42, early_stopping=True, n_iter_no_change=10)
mlp.fit(X_tr, y_tr)
acc_train = mlp.score(X_tr, y_tr)
acc_test  = mlp.score(X_te, y_te)
print(f"Accuracy entrenamiento: {acc_train:.3f}")
print(f"Accuracy test:          {acc_test:.3f}")

## 5. Curva de Umbral

El **umbral** $p_{th}$ es la tasa de error físico por debajo de la cual aumentar el tamaño del código reduce el error lógico. Para el surface code $p_{th} \approx 1\%$ bajo ruido de Pauli independiente.

In [ ]:
def logical_error_rate(n_code, p_phys, n_trials=3000, decoder='lookup'):
    """
    Estima tasa de error lógico para un código de repetición de longitud n_code.
    Usa decodificador de lookup (exacto para repetición).
    """
    H = np.zeros((n_code-1, n_code), dtype=int)
    for i in range(n_code-1):
        H[i, i] = 1; H[i, i+1] = 1
    
    logical_errors = 0
    for _ in range(n_trials):
        error = apply_random_error(n_code, p_phys)
        syndrome = compute_syndrome(H, error)
        correction = mwpm_decode(H, syndrome)
        residual = (error + correction) % 2
        if np.sum(residual) % 2 == 1:  # error lógico
            logical_errors += 1
    return logical_errors / n_trials

p_vals = np.linspace(0.01, 0.20, 15)
code_sizes = [3, 5, 7, 9]
colors = ['steelblue', 'darkorange', 'mediumseagreen', 'crimson']

print("Calculando curvas de umbral...")
p_L_curves = {}
for n in code_sizes:
    p_L_curves[n] = [logical_error_rate(n, p, n_trials=2000) for p in p_vals]
    print(f"  n={n}: completado")

fig, ax = plt.subplots(figsize=(8, 5))
for n, col in zip(code_sizes, colors):
    ax.semilogy(p_vals * 100, p_L_curves[n], 'o-', color=col, lw=2, label=f'n = {n}')

ax.axvline(10, color='black', ls='--', lw=1.5, label='~10% umbral rep.')
ax.set_xlabel('Tasa de error físico p (%)'); ax.set_ylabel('Tasa de error lógico P_L')
ax.set_title('Código de Repetición — Curva de Umbral (MWPM)')
ax.legend(); ax.grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig('threshold_curve.png', dpi=100)
plt.show()
print("Curva de umbral guardada.")

## 6. Comparativa de Overhead: Surface Code vs qLDPC

| Código | [[n, k, d]] ejemplo | Qubits físicos por lógico (n/k) | Distancia |
|--------|--------------------|---------------------------------|-----------|
| Surface code d=7 | [[98, 1, 7]] | 98 | 7 |
| Surface code d=13 | [[338, 1, 13]] | 338 | 13 |
| Bivariate Bicycle | [[72, 12, 6]] | 6 | 6 |
| Bivariate Bicycle | [[144, 12, 12]] | 12 | 12 |

**Factor de mejora**: 338/12 ≈ **28×** menos qubits físicos por qubit lógico comparable.

In [ ]:
import pandas as pd

codes = [
    {'Código': 'Surface d=3', 'n': 18,  'k': 1,  'd': 3,  'n/k': 18},
    {'Código': 'Surface d=5', 'n': 50,  'k': 1,  'd': 5,  'n/k': 50},
    {'Código': 'Surface d=7', 'n': 98,  'k': 1,  'd': 7,  'n/k': 98},
    {'Código': 'Surface d=13','n': 338, 'k': 1,  'd': 13, 'n/k': 338},
    {'Código': 'BB [[72,12,6]]', 'n': 72,  'k': 12, 'd': 6,  'n/k': 6},
    {'Código': 'BB [[144,12,12]]','n': 144,'k': 12, 'd': 12, 'n/k': 12},
    {'Código': 'HP [[n=800,k=25,d≥20]]','n': 800,'k': 25,'d': 20, 'n/k': 32},
]

df = pd.DataFrame(codes)
print(df.to_string(index=False))

fig2, ax2 = plt.subplots(figsize=(8, 4))
x = range(len(codes))
bars = ax2.bar(x, df['n/k'], color=['steelblue']*4 + ['darkorange']*3)
ax2.axhline(12, color='red', ls='--', lw=1.5, label='Target: 12 qubits/lógico')
ax2.set_xticks(x); ax2.set_xticklabels(df['Código'], rotation=25, ha='right', fontsize=8)
ax2.set_ylabel('Qubits físicos por qubit lógico (n/k)')
ax2.set_title('Overhead: Surface Code vs qLDPC')
ax2.legend()
from matplotlib.patches import Patch
ax2.legend(handles=[
    Patch(color='steelblue', label='Surface Code'),
    Patch(color='darkorange', label='qLDPC (Bivariate Bicycle / HP)'),
    plt.Line2D([0],[0], color='red', ls='--', label='Target 12×')
])
plt.tight_layout()
plt.show()
print("\nMejora máxima: Surface d=13 → BB [[144,12,12]]: factor", 338//12, "×")